# Multi-Tool AI Agent for Bangladesh

A LangChain agent that answers data questions from three Bangladesh datasets using SQLite tools, and general knowledge questions using a web search tool.

| Tool | Data |
|---|---|
| `InstitutionsDBTool` | `institutions.db` (schools, colleges, madrasas) |
| `HospitalsDBTool` | `hospitals.db` (hospitals, clinics, health facilities) |
| `RestaurantsDBTool` | `restaurants.db` (restaurants, ratings, reviews) |
| `WebSearchTool` | Tavily web search (DuckDuckGo if no Tavily key) |

GitHub: https://github.com/Jahid15/bangladesh-multi-tool-agent

## 1. Setup

In [1]:
import os, sys

if "google.colab" in sys.modules:
    if not os.path.exists("bangladesh-multi-tool-agent"):
        !git clone -q https://github.com/Jahid15/bangladesh-multi-tool-agent.git
    %cd bangladesh-multi-tool-agent
    !pip install -q -r requirements.txt

## 2. API keys

In Colab, add `ANTHROPIC_API_KEY` and (optional) `TAVILY_API_KEY` in the **Secrets** panel (key icon on the left sidebar). Otherwise you will be asked to paste them.

In [2]:
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()

def load_key(name, required=True):
    if os.getenv(name):
        return
    try:
        from google.colab import userdata
        os.environ[name] = userdata.get(name)
        return
    except Exception:
        pass
    value = getpass(f"{name}{'' if required else ' (optional, press Enter to skip)'}: ")
    if value:
        os.environ[name] = value

load_key("ANTHROPIC_API_KEY")
load_key("TAVILY_API_KEY", required=False)
print("Anthropic key set:", bool(os.getenv("ANTHROPIC_API_KEY")))
print("Tavily key set:", bool(os.getenv("TAVILY_API_KEY")))

Anthropic key set: True
Tavily key set: True


## 3. SQLite databases

The CSV datasets are converted to SQLite by `build_databases.py`. The databases are already in the repo. Run `!python build_databases.py` to rebuild them.

In [3]:
import sqlite3

for name in ["institutions", "hospitals", "restaurants"]:
    with sqlite3.connect(f"databases/{name}.db") as conn:
        rows = conn.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
        columns = conn.execute(f"PRAGMA table_info({name})").fetchall()
    print(f"{name}.db -> table '{name}' ({rows:,} rows)")
    print("  " + ", ".join(f"{c[1]} {c[2]}" for c in columns) + "\n")

institutions.db -> table 'institutions' (34,897 rows)
  eiin INTEGER, name TEXT, type TEXT, education_level TEXT, management_type TEXT, student_type TEXT, affiliation TEXT, mpo_status TEXT, division TEXT, district TEXT, thana TEXT, union_name TEXT, mauza TEXT, area_status TEXT, geographical_status TEXT, address TEXT, post_office TEXT, mobile TEXT

hospitals.db -> table 'hospitals' (38,886 rows)
  id INTEGER, name TEXT, name_bangla TEXT, code INTEGER, agency TEXT, type TEXT, is_private INTEGER, division TEXT, district TEXT, city_corporation TEXT, upazila TEXT, paurasava TEXT, union_name TEXT

restaurants.db -> table 'restaurants' (10,755 rows)
  place_id TEXT, name TEXT, address TEXT, rating REAL, number_of_reviews INTEGER, affluence INTEGER, latitude REAL, longitude REAL



## 4. Test the tools directly

In [4]:
from tools import get_tools

institutions_tool, hospitals_tool, restaurants_tool, web_search_tool = get_tools()

print(hospitals_tool.invoke("SELECT district, COUNT(*) AS total FROM hospitals GROUP BY district ORDER BY total DESC LIMIT 5"))
print()
print(institutions_tool.invoke("SELECT type, COUNT(*) AS total FROM institutions GROUP BY type ORDER BY total DESC"))
print()
print(restaurants_tool.invoke("SELECT name, rating, number_of_reviews FROM restaurants WHERE address LIKE '%Dhaka%' ORDER BY number_of_reviews DESC LIMIT 5"))
print()
print(web_search_tool.invoke("Directorate General of Health Services Bangladesh")[:600])

district | total
Dhaka | 2919
Cumilla | 1581
Chattogram | 1472
Mymensingh | 1180
Tangail | 1082

type | total
School | 19230
Madrasha | 9314
College | 3190
Technical and Vocational | 1940
School and College | 1223

name | rating | number_of_reviews
Hotel Noorjahan | 3.5 | 17655
Star Kebab & Restaurant | 4.2 | 16992
Domino's Pizza - Dhanmondi | 4.3 | 8276
Hotel Al Razzak Restaurant | 3.9 | 7513
Hirajheel Hotel | 4.1 | 5626



Directorate General of Health Services (Bangladesh) - Wikipedia
https://en.wikipedia.org/wiki/Directorate_General_of_Health_Services_(Bangladesh)
The Directorate General of Health Services (DGHS) (Bengali: স্বাস্থ্য অধিদপ্তর) is a Bangladeshi government directorate under the Ministry of Health and Family Welfare "Ministry of Health and Family Welfare (Bangladesh)") responsible for health services in Bangladesh.

## History

[edit&action=edit&section=1 "Edit section: History")]

The DGHS was established as a directorate in 1978. It was upgraded to a directorate general in 1980. [...] Abul Basha


## 5. Main agent

In [5]:
from agent import build_agent, to_text

executor = build_agent(verbose=False)
executor.return_intermediate_steps = True

def demo(question):
    result = executor.invoke({"input": question})
    print("Question:", question)
    for action, _ in result["intermediate_steps"]:
        print(f"Tool: {action.tool}")
        print(f"Input: {action.tool_input}")
    print("\nAnswer:", to_text(result["output"]))

### Data and statistics queries (routed to DB tools)

In [6]:
demo("How many hospitals are in Dhaka district?")

Question: How many hospitals are in Dhaka district?
Tool: HospitalsDBTool
Input: SELECT COUNT(*) AS total FROM hospitals WHERE district = 'Dhaka';

Answer: There are 2,919 hospitals and other health facilities recorded in Dhaka district in the database. This count covers all facility types in the hospitals dataset, including clinics, diagnostic centers, and hospitals, both public and private.


In [7]:
demo("How many private hospitals and clinics are in Chattogram division?")

Question: How many private hospitals and clinics are in Chattogram division?
Tool: HospitalsDBTool
Input: SELECT COUNT(*) AS total FROM hospitals WHERE is_private = 1 AND division LIKE '%Chattogram%';

Answer: There are **3,177** private hospitals and clinics in Chattogram division. This counts every facility flagged as private in the database, including diagnostic centers and other private health facilities, not only hospitals and clinics.


In [8]:
demo("Which district has the most madrasas?")

Question: Which district has the most madrasas?
Tool: InstitutionsDBTool
Input: SELECT district, COUNT(*) AS madrasa_count FROM institutions WHERE type = 'Madrasha' GROUP BY district ORDER BY madrasa_count DESC LIMIT 5;

Answer: **Comilla** has the most madrasas in the database, with 386. Mymensingh is close behind with 385, followed by Chittagong (340), Bogra (309) and Jessore (308).


In [9]:
demo("How many government colleges are in Rajshahi division?")

Question: How many government colleges are in Rajshahi division?
Tool: InstitutionsDBTool
Input: SELECT COUNT(*) AS count FROM institutions WHERE type = 'College' AND management_type = 'GOVERNMENT' AND division = 'Rajshahi';

Answer: There are **42 government colleges** in Rajshahi division, based on the institutions database.


In [10]:
demo("List the top 5 rated restaurants in Dhaka with at least 100 reviews.")

Question: List the top 5 rated restaurants in Dhaka with at least 100 reviews.
Tool: RestaurantsDBTool
Input: SELECT name, address, rating, number_of_reviews FROM restaurants WHERE address LIKE '%Dhaka%' AND number_of_reviews >= 100 ORDER BY rating DESC, number_of_reviews DESC LIMIT 5;

Answer: Here are the top 5 rated restaurants in Dhaka with at least 100 reviews:

1. **Elements Restaurant, InterContinental Dhaka**: 4.6 stars (462 reviews), at 1 Minto Rd, Dhaka
2. **Govinda's Sweets & Restaurant**: 4.6 stars (146 reviews), at Ln. No. 2, Dhaka 1216
3. **Yum Cha District** (BFC Building, Road No. 11): 4.5 stars (2,814 reviews)
4. **Yum Cha District** (Dhanmondi): 4.5 stars (2,277 reviews)
5. **Izumi Japanese Kitchen**: 4.5 stars (1,425 reviews), at House 88, Road 113/A, Dhaka 1212

Ties in rating were broken by number of reviews. The two Yum Cha District branches are listed separately because they are different locations.


### General knowledge queries (routed to WebSearchTool)

In [11]:
demo("What is the role of DGHS in Bangladesh?")

Question: What is the role of DGHS in Bangladesh?
Tool: WebSearchTool
Input: role of DGHS Directorate General of Health Services Bangladesh

Answer: The Directorate General of Health Services (DGHS) is a government body under the Ministry of Health and Family Welfare that is responsible for public health services across Bangladesh. Its main responsibilities are:

- **Delivering health services at all levels** of the country, from the national level down to districts and upazilas.
- **Managing health services** and running the government's health programmes.
- **Planning and implementing policies** through its administrative structure, including preparing action plans.
- **Advising the ministry** with technical guidance on healthcare guidelines and public health matters.

It is headed by a Director General, who is assisted by Additional Directors General, with line directors and other staff running the work. Sources differ on its founding date: one says it was established as a directora

In [12]:
demo("What is the healthcare policy in Bangladesh?")

Question: What is the healthcare policy in Bangladesh?
Tool: WebSearchTool
Input: healthcare policy in Bangladesh

Answer: Bangladesh's healthcare policy aims to provide universal access to health services through its public system. The main reference points are:

- **National Health Policy 2011:** This sets out 15 goals, including reducing child and maternal mortality, improving reproductive health and family planning services, and meeting sustainable development targets.
- **National Health Compact (recent):** The government commits to universal health coverage, organised around six pillars. These include modernising health infrastructure, strengthening primary health care (staffing, diagnostics and essential supplies), improving access to essential medicines, expanding immunisation, increasing facility-based deliveries, reducing financial hardship from healthcare costs, and making health facilities more climate-resilient. The target is universal coverage by 2032.

The public system 

## 6. Ask your own question

In [13]:
demo("How many community clinics are in Sylhet division?")

Question: How many community clinics are in Sylhet division?
Tool: HospitalsDBTool
Input: SELECT COUNT(*) AS community_clinics FROM hospitals WHERE type = 'Community Clinic' AND division = 'Sylhet';

Answer: There are **952 community clinics** in Sylhet division, according to the hospitals database.
